# Price Window Model Training
## Training from 2010-2025 Maize Prices Data

In [11]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# Load the Excel file
excel_path = Path('2010 - 2025 Maize Prices.xlsx')
df = pd.read_excel(excel_path)

print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
print('Data types:')
print(df.dtypes)

Columns: ['Year', 'Week', 'District', 'Season', 'Fuel_price_Rs_per_L', 'Rainfall_mm', 'Temp_C', 'Festival', 'Import_tax_Rs_per_kg', 'demand_Festival', 'demand_tax', 'demand_weather', 'demand_season', 'demand_fuel', 'Farm_Gate_Price_Rs_per_kg']
Shape: (2496, 15)
Data types:
Year                           int64
Week                           int64
District                         str
Season                           str
Fuel_price_Rs_per_L            int64
Rainfall_mm                  float64
Temp_C                       float64
Festival                         str
Import_tax_Rs_per_kg           int64
demand_Festival              float64
demand_tax                   float64
demand_weather               float64
demand_season                float64
demand_fuel                  float64
Farm_Gate_Price_Rs_per_kg    float64
dtype: object


In [12]:
# Inspect the data more closely
print('Sample data:')
print(df.head())
print('\nDistricts:', df['District'].unique())
print('Weeks range:', df['Week'].min(), '-', df['Week'].max())
print('Years range:', df['Year'].min(), '-', df['Year'].max())
print('Price range: Rs', df['Farm_Gate_Price_Rs_per_kg'].min(), '-', df['Farm_Gate_Price_Rs_per_kg'].max())

Sample data:
   Year  Week       District Season  Fuel_price_Rs_per_L  Rainfall_mm  Temp_C  \
0  2010     1   Anuradhapura   Maha                   73         1.13   25.73   
1  2010     1     Monaragala   Maha                   73         8.50   24.77   
2  2010     1  Tissamaharama   Maha                   73         0.23   26.13   
3  2010     2   Anuradhapura   Maha                   73         2.87   25.43   
4  2010     2     Monaragala   Maha                   73         2.11   24.20   

  Festival  Import_tax_Rs_per_kg  demand_Festival  demand_tax  demand_weather  \
0      YES                    10              NaN         NaN             NaN   
1      YES                    10              NaN         NaN             NaN   
2      YES                    10              NaN         NaN             NaN   
3      YES                    10              NaN         NaN             NaN   
4      YES                    10              NaN         NaN             NaN   

   demand_sea

In [16]:
# IMPROVED: Process data with better statistics and label distribution
# Step 1: Get baseline statistics (aggregated across all years)
baseline = df.groupby(['District', 'Week'])['Farm_Gate_Price_Rs_per_kg'].agg([
    ('baseline_price', 'mean'),
    ('baseline_std', 'std'),
    ('years_with_data', 'count')
]).reset_index()

# Step 2: Get year-specific data with variation
yearly = df.groupby(['District', 'Year', 'Week'])['Farm_Gate_Price_Rs_per_kg'].agg([
    ('price', 'mean'),
]).reset_index()

# Step 3: Merge to get both baseline and yearly values
grouped = yearly.merge(baseline, left_on=['District', 'Week'], right_on=['District', 'Week'])
grouped = grouped.rename(columns={
    'District': 'Location',
    'Week': 'WeekNum',
    'price': 'avg_price',
    'baseline_price': 'baseline_avg_price',
    'baseline_std': 'baseline_std_price'
})

# For median and std_price at year level, calculate across baseline
grouped['median_price'] = grouped['avg_price']  # Single year = price is its own median
grouped['std_price'] = grouped['baseline_std_price']  # Use baseline std as reference
grouped['years_count'] = 1  # Each row is 1 year

print(f'Total rows: {len(grouped)}')
print(f'Locations: {grouped["Location"].nunique()}')
print(f'Years: {grouped["Year"].nunique()}')

# Get overall statistics for normalization
overall_min = df['Farm_Gate_Price_Rs_per_kg'].min()
overall_max = df['Farm_Gate_Price_Rs_per_kg'].max()
overall_mean = df['Farm_Gate_Price_Rs_per_kg'].mean()
overall_std = df['Farm_Gate_Price_Rs_per_kg'].std()

print(f'\nOverall statistics:')
print(f'  Min: {overall_min:.2f}')
print(f'  Max: {overall_max:.2f}')
print(f'  Mean: {overall_mean:.2f}')
print(f'  Std: {overall_std:.2f}')

# Normalized average price (0-1)
grouped['avg_norm'] = (grouped['avg_price'] - overall_min) / (overall_max - overall_min)

# Risk normalization (volatility relative to baseline)
max_baseline_std = grouped['baseline_std_price'].max()
grouped['risk_norm'] = grouped['baseline_std_price'] / max_baseline_std

# HighPriceScore based on percentile ranking
grouped['HighPriceScore'] = grouped['avg_price'].rank(pct=True)

# IMPROVED: Better label assignment using multiple criteria
def smart_label(row):
    """
    Assign label based on:
    1. Position in price distribution
    2. Stability (lower risk = higher confidence)
    """
    price_percentile = row['HighPriceScore']
    
    if price_percentile >= 0.65:
        return 'STRONG'
    elif price_percentile <= 0.35:
        return 'WEAK'
    else:
        return 'MODERATE'

grouped['Label'] = grouped.apply(smart_label, axis=1)

# IMPROVED: Confidence score (0-100)
# Higher confidence = price is closer to mean AND lower volatility
grouped['conf_score'] = (
    (1 - abs(grouped['avg_price'] - overall_mean) / (overall_max - overall_min)) * 0.6 +
    (1 - grouped['risk_norm']) * 0.4
).clip(0, 1)

# Convert to percentage
grouped['conf_score'] = grouped['conf_score'] * 100

# Confidence level (categorical)
def confidence_level(score):
    if score >= 70:
        return 'High'
    elif score >= 40:
        return 'Medium'
    else:
        return 'Low'

grouped['Confidence'] = grouped['conf_score'].apply(confidence_level)

print('\n=== IMPROVED MODEL STATISTICS ===')
print(f'\nLabel distribution:')
print(grouped['Label'].value_counts())
print(f'\nConfidence distribution:')
print(grouped['Confidence'].value_counts())
print(f'\nPrice by Label:')
print(grouped.groupby('Label')['avg_price'].agg(['min', 'max', 'mean', 'count']))
print(f'\nModel preview:')
print(grouped.head(15))

Total rows: 2496
Locations: 3
Years: 16

Overall statistics:
  Min: 11.80
  Max: 271.67
  Mean: 72.42
  Std: 48.25

=== IMPROVED MODEL STATISTICS ===

Label distribution:
Label
WEAK        877
STRONG      874
MODERATE    745
Name: count, dtype: int64

Confidence distribution:
Confidence
Medium    2201
High       242
Low         53
Name: count, dtype: int64

Price by Label:
                min         max        mean  count
Label                                             
MODERATE  42.562500   66.255556   54.082065    745
STRONG    66.259259  271.666667  127.377220    874
WEAK      11.800000   42.500000   33.223210    877

Model preview:
        Location  Year  WeekNum  avg_price  baseline_avg_price  \
0   Anuradhapura  2010        1  40.000000           68.349836   
1   Anuradhapura  2010        2  41.000000           70.255571   
2   Anuradhapura  2010        3  41.000000           65.682542   
3   Anuradhapura  2010        4  40.500000           64.531875   
4   Anuradhapura  2010 

In [17]:
# Save improved model with better distribution
model_output = grouped[[
    'Location', 'Year', 'WeekNum', 'avg_price', 'median_price', 'std_price', 'years_count',
    'avg_norm', 'risk_norm', 'HighPriceScore', 'Label', 'conf_score', 'Confidence'
]].copy()

# Round numeric columns for cleaner output
numeric_cols = ['avg_price', 'median_price', 'std_price', 'avg_norm', 'risk_norm', 'HighPriceScore', 'conf_score']
for col in numeric_cols:
    model_output[col] = model_output[col].round(6)

# Save the model
output_path = Path('high_price_season_model.csv')
model_output.to_csv(output_path, index=False)

print(f'✅ IMPROVED MODEL SAVED')
print(f'   Path: {output_path}')
print(f'   Total rows: {len(model_output)}')
print(f'   Locations: {model_output["Location"].nunique()}')
print(f'   Years: {model_output["Year"].nunique()}')
print(f'\n📊 MODEL QUALITY:')
print(f'   STRONG labels: {(model_output["Label"] == "STRONG").sum()} ({(model_output["Label"] == "STRONG").sum() / len(model_output) * 100:.1f}%)')
print(f'   MODERATE labels: {(model_output["Label"] == "MODERATE").sum()} ({(model_output["Label"] == "MODERATE").sum() / len(model_output) * 100:.1f}%)')
print(f'   WEAK labels: {(model_output["Label"] == "WEAK").sum()} ({(model_output["Label"] == "WEAK").sum() / len(model_output) * 100:.1f}%)')
print(f'\n   High confidence: {(model_output["Confidence"] == "High").sum()} ({(model_output["Confidence"] == "High").sum() / len(model_output) * 100:.1f}%)')
print(f'   Medium confidence: {(model_output["Confidence"] == "Medium").sum()} ({(model_output["Confidence"] == "Medium").sum() / len(model_output) * 100:.1f}%)')
print(f'   Low confidence: {(model_output["Confidence"] == "Low").sum()} ({(model_output["Confidence"] == "Low").sum() / len(model_output) * 100:.1f}%)')
print(f'\n✨ Sample data:')
print(model_output.head(20))

✅ IMPROVED MODEL SAVED
   Path: high_price_season_model.csv
   Total rows: 2496
   Locations: 3
   Years: 16

📊 MODEL QUALITY:
   STRONG labels: 874 (35.0%)
   MODERATE labels: 745 (29.8%)
   WEAK labels: 877 (35.1%)

   High confidence: 242 (9.7%)
   Medium confidence: 2201 (88.2%)
   Low confidence: 53 (2.1%)

✨ Sample data:
        Location  Year  WeekNum  avg_price  median_price  std_price  \
0   Anuradhapura  2010        1  40.000000     40.000000  44.649710   
1   Anuradhapura  2010        2  41.000000     41.000000  45.115887   
2   Anuradhapura  2010        3  41.000000     41.000000  43.710043   
3   Anuradhapura  2010        4  40.500000     40.500000  42.670715   
4   Anuradhapura  2010        5  35.333333     35.333333  36.419725   
5   Anuradhapura  2010        6  26.000000     26.000000  40.634354   
6   Anuradhapura  2010        7  24.666667     24.666667  38.078129   
7   Anuradhapura  2010        8  24.500000     24.500000  39.429288   
8   Anuradhapura  2010        9 